# Fundamentos e exploração dos dados

Percurso dos capítulos 1–6: ler tabelas, inspecionar medidas, avaliar previsões pequenas e explorar grupos e componentes principais. Os dados reais usados aqui são apenas os pinguins de desenvolvimento.

Execute as células pela ordem apresentada. No Colab, use o ZIP completo dos materiais de apoio; no computador, mantenha este caderno na pasta principal. Guarde as suas interpretações em células de texto. As saídas são calculadas quando executar o caderno.


## Preparar os ficheiros


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import zipfile

# No computador, abra este caderno junto de reproduzir.py.
# No Colab, a primeira execução pede o ZIP completo dos materiais.
candidatos = [Path.cwd(), Path.cwd().parent]
pasta_colab = Path('/content/materiais_biociencias')
if pasta_colab.exists():
    candidatos += [p.parent for p in pasta_colab.rglob('reproduzir.py')]
raiz = next((p for p in candidatos if (p / 'reproduzir.py').is_file()), None)
if raiz is None:
    try:
        from google.colab import files
    except ImportError as erro:
        raise FileNotFoundError(
            'Abra o caderno na pasta que contém reproduzir.py, dados e reproducao.'
        ) from erro
    carregados = files.upload()
    arquivos = [Path(nome) for nome in carregados if nome.lower().endswith('.zip')]
    if len(arquivos) != 1:
        raise ValueError('Carregue um único ZIP completo dos materiais de apoio.')
    pasta_colab.mkdir(parents=True, exist_ok=True)
    destino = pasta_colab.resolve()
    with zipfile.ZipFile(arquivos[0]) as arquivo:
        for item in arquivo.infolist():
            caminho = (destino / item.filename).resolve()
            if not caminho.is_relative_to(destino):
                raise ValueError('O ZIP contém um caminho fora da pasta de destino.')
        arquivo.extractall(destino)
    encontrados = list(destino.rglob('reproduzir.py'))
    if len(encontrados) != 1:
        raise ValueError('O ZIP deve conter uma única pasta completa dos materiais.')
    raiz = encontrados[0].parent
os.chdir(raiz)
print('Pasta de trabalho:', raiz.name)

def executar_python(script, *argumentos):
    subprocess.run([sys.executable, script, *argumentos], check=True, cwd=raiz)


## Conferir o ambiente

Esta célula instala as dependências apenas se necessário. Se pedir um reinício, reinicie o ambiente e volte à primeira célula.


In [ ]:
from importlib.metadata import version, PackageNotFoundError

requisitos = {}
for linha in Path('requisitos.txt').read_text().splitlines():
    if '==' in linha and not linha.lstrip().startswith('#'):
        nome, esperada = linha.strip().split('==', 1)
        requisitos[nome] = esperada
alterar = False
for nome, esperada in requisitos.items():
    try:
        atual = version(nome)
    except PackageNotFoundError:
        atual = 'não instalada'
    print(f'{nome}: {atual} (referência: {esperada})')
    alterar = alterar or atual != esperada
if alterar:
    ja_importadas = any(n in sys.modules for n in ['numpy', 'pandas', 'sklearn', 'matplotlib', 'scipy'])
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requisitos.txt'], check=True)
    if ja_importadas:
        raise RuntimeError(
            'Dependências instaladas. Reinicie o ambiente e execute desde a primeira célula '
            'para usar as versões agora instaladas.'
        )
else:
    print('O ambiente já tem as versões de referência.')


## 1. Ler os pinguins

Confirme o significado de cada linha e os nomes das medidas.


In [ ]:
import pandas as pd
from IPython.display import display, Image
dados = pd.read_csv('dados/penguins_desenvolvimento.csv')
print('Dimensão:', dados.shape)
display(dados.head())


## 2. Executar os capítulos 1–4

O programa guarda as tabelas e os exemplos fictícios identificados em `reproducao/fundamentos/resultados`. Consulte o script para adaptar as operações.


In [ ]:
executar_python('reproducao/fundamentos/reproduzir_fundamentos.py', '--capitulos', '1', '2', '3', '4')


In [ ]:
pasta = Path('reproducao/fundamentos/resultados')
display(pd.read_csv(pasta / 'tabelas/cap02_resumos_medidas.csv'))
display(Image(filename=str(pasta / 'figuras/u2_hist_massa.png')))


## 3. Explorar grupos e componentes

A espécie é consultada depois do ajuste para interpretar os grupos e as coordenadas. Não participa no cálculo.


In [ ]:
executar_python('reproducao/fundamentos/reproduzir_fundamentos.py', '--capitulos', '5', '6')


In [ ]:
display(pd.read_csv(pasta / 'tabelas/cap05_comparacao_K.csv'))
display(pd.read_csv(pasta / 'tabelas/cap05_perfis_K3.csv'))
display(Image(filename=str(pasta / 'figuras/u5_kmeans_grupos.png')))
display(pd.read_csv(pasta / 'tabelas/cap06_variabilidade.csv'))
display(Image(filename=str(pasta / 'figuras/u6_acp_scores.png')))


## Interpretar

Explique o efeito da escala nas distâncias e o significado da variabilidade representada pelas duas primeiras componentes. Consulte o README do percurso para localizar os restantes resultados. Uma mudança do sinal de uma componente, aplicada também às suas coordenadas, não altera a direção representada.
